# 04 · Kenya → Ghana transfer (step 10) and fairness (step 11)

**Part 1, transfer (step 10):** Alice. **Decision record:** [ADR 0019](../docs/adr/0019-kenya-to-ghana-transfer.md). **Plain write-up:** [learnings/10](../learnings/10_transfer.md).
**Part 2, fairness (step 11):** Liza, added below Part 1.

**The question.** NkwaPay's labelled loans are all Kenyan. Its Ghanaian applicants have **no repayment outcomes**. How different is Ghana, can the Kenyan models be adjusted for it, and what should NkwaPay see before it trusts them there?

**What we can and can't measure.**
- **Default model (Task 2):** Ghana has no labels, so we can only compare *who applies* and *how the model scores them*, and use Kenyan stand-ins.
- **Amount model (Task 1):** `Test.csv` contains the real amount of every Ghanaian loan, so here we **can** score the Kenyan model on Ghana directly.

**Privacy:** every output is a count, a share, a summary or a score. No applicant's row is printed.

In [1]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
from src.features import build, FEATURES, preprocessor, splits
from src.amounts import usd_rate, AMOUNT_CLUES, regressor
from src.classification_experiments import final_model, logit

TABLES, FIGURES = ROOT / "learnings" / "tables", ROOT / "learnings" / "figures"
DATA = ROOT / "data" / "zindi"
tr, te = build(str(DATA))
gh, ke_test = te[te.country_id == "Ghana"].copy(), te[te.country_id == "Kenya"].copy()
for d in (tr, gh, ke_test):
    d["usd_rate"] = usd_rate(d, DATA)
CUTOFF = {"Kenya": 0.021, "Ghana": 0.032}   # ADR 0007, on calibrated probabilities only
print(len(tr), "labelled Kenyan loans |", len(ke_test), "Kenyan test loans |", len(gh), "Ghanaian test loans")

68464 labelled Kenyan loans | 15069 Kenyan test loans | 3525 Ghanaian test loans


## 1. How different are Ghana's applicants?

We put three groups side by side:
- **Kenya, learned from:** the labelled loans (`Train.csv`);
- **Kenya, test:** Kenyan applicants in `Test.csv`. This is a control: if the method works, it should find these close to the training loans;
- **Ghana:** Ghanaian applicants in `Test.csv`.

Amounts are shown in local currency and in US dollars (that year's average rate).

In [2]:
def profile(d):
    usd = d.Total_Amount / d.usd_rate
    seen_type, seen_lender = d.loan_type.isin(set(tr.loan_type)), d.lender_id.isin(set(tr.lender_id))
    return pd.Series({
        "loans": len(d), "customers": d.customer_id.nunique(),
        "also a customer in Kenya's labelled loans": d.customer_id.isin(set(tr.customer_id)).mean(),
        "most common loan type": f"{d.loan_type.value_counts().index[0]} ({d.loan_type.value_counts(normalize=True).iloc[0]:.0%})",
        "loan type never seen in Kenya's labelled loans": 1 - seen_type.mean(),
        "lender never seen in Kenya's labelled loans": 1 - seen_lender.mean(),
        "7-day loans": (d.duration == 7).mean(), "loans longer than 90 days": (d.duration > 90).mean(),
        "flagged 'New Loan'": (d.New_versus_Repeat == "New Loan").mean(),
        "first loan in the data (no earlier loans)": (d.n_prev_loans == 0).mean(),
        "earlier loans, but no known outcome": ((d.n_prev_loans > 0) & (d.no_known_history == 1)).mean(),
        "at least one known earlier outcome": (d.no_known_history == 0).mean(),
        "earlier loans (median)": d.n_prev_loans.median(),
        "amount, local currency (median)": d.Total_Amount.median(),
        "amount, US$ (median)": usd.median(), "amount, US$ (25%-75%)": f"{usd.quantile(.25):,.0f}-{usd.quantile(.75):,.0f}",
        "last earlier amount, US$ (median)": (d.last_amount / d.usd_rate).median(),
        "dates": f"{d.disbursement_date.min():%b %Y}-{d.disbursement_date.max():%b %Y}",
    })
prof = pd.DataFrame({"Kenya, learned from": profile(tr), "Kenya, test": profile(ke_test), "Ghana": profile(gh)})
prof.to_csv(TABLES / "10_profile.csv")
prof

,"Kenya, learned from","Kenya, test",Ghana
loans,68464,15069,3525
customers,6508,4516,446
also a customer in Kenya's labelled loans,1.0,0.983874,0.0
most common loan type,Type_1 (90%),Type_1 (90%),Type_3 (86%)
loan type never seen in Kenya's labelled loans,0.0,0.000199,0.872624
lender never seen in Kenya's labelled loans,0.0,0.0,0.967092
7-day loans,0.9466,0.950693,0.862128
loans longer than 90 days,0.002162,0.001725,0.135887
flagged 'New Loan',0.008267,0.006902,0.01844
first loan in the data (no earlier loans),0.092939,0.090981,0.187801


**Loan types, in detail.** For each type: its share of Ghanaian loans, how many Kenyan labelled loans it has, and the Kenyan default rate. A type with few Kenyan loans gives the model very little to learn from.

In [3]:
kt = tr.groupby("loan_type").agg(kenya_loans=("target", "size"), kenya_default_rate=("target", "mean"))
types = (gh.loan_type.value_counts(normalize=True).rename("share of Ghana").to_frame()
         .join(gh.groupby("loan_type").Total_Amount.median().div(gh.usd_rate.median()).rename("Ghana median US$"))
         .join(kt).fillna({"kenya_loans": 0}))
types.to_csv(TABLES / "10_loan_types.csv")
types.round(3)

,share of Ghana,Ghana median US$,kenya_loans,kenya_default_rate
loan_type,,,,
Type_3,0.862,295.629,0.0,NaN
Type_2,0.127,9304.463,74.0,0.608
Type_8,0.010,6220.473,0.0,NaN


**How far has each clue moved?** The **population stability index (PSI)** is the standard drift score in credit scoring. It splits a clue into bins (one per value for types and flags; for numbers, the tenths of both countries), then compares each bin's share in Kenya and in Ghana. The usual reading:
- below 0.1: stable;
- 0.1 to 0.25: some shift;
- above 0.25: a big shift, so don't trust the model on it without checking.

The money clues are scored twice: raw (cedis vs shillings) and in dollars.

In [4]:
def psi(k, g, categorical=False, eps=1e-4):
    if categorical or k.nunique() <= 20:   # few distinct values (flags, counts like n_defaults): one bin per value
        cats = sorted(set(k.astype(str)) | set(g.astype(str)))
        pk = k.astype(str).value_counts(normalize=True).reindex(cats, fill_value=0)
        pg = g.astype(str).value_counts(normalize=True).reindex(cats, fill_value=0)
    else:
        # bin edges = the tenths of Kenya AND of Ghana, so a shift is visible even when most Kenyan values are equal
        # (95% of Kenyan loans are 7 days long, so Kenya's tenths alone would put 7 and 180 days in one bin)
        edges = np.unique(np.concatenate([np.nanquantile(k, np.linspace(0, 1, 11)), np.nanquantile(g, np.linspace(0, 1, 11))]))
        bins = lambda x: pd.Series(np.where(np.isnan(x), -1, np.searchsorted(edges[1:-1], x, side="right")))
        pk, pg = bins(k.to_numpy(float)).value_counts(normalize=True), bins(g.to_numpy(float)).value_counts(normalize=True)
        pk, pg = pk.reindex(pk.index.union(pg.index), fill_value=0), pg.reindex(pk.index.union(pg.index), fill_value=0)
    pk, pg = pk + eps, pg + eps
    return float(((pg - pk) * np.log(pg / pk)).sum())

rows = []
for c in FEATURES:
    cat = c in ["loan_type", "New_versus_Repeat"]
    r = {"clue": c, "PSI Kenya test": psi(tr[c], ke_test[c], cat), "PSI Ghana": psi(tr[c], gh[c], cat)}
    if c in AMOUNT_CLUES:
        r["PSI Ghana, in US$"] = psi(tr[c] / tr.usd_rate, gh[c] / gh.usd_rate)
    rows.append(r)
r = {"clue": "Total_Amount (Task 1 target)", "PSI Kenya test": psi(tr.Total_Amount, ke_test.Total_Amount),
     "PSI Ghana": psi(tr.Total_Amount, gh.Total_Amount), "PSI Ghana, in US$": psi(tr.Total_Amount / tr.usd_rate, gh.Total_Amount / gh.usd_rate)}
drift = pd.DataFrame(rows + [r]).set_index("clue")
drift["reading (Ghana)"] = pd.cut(drift[["PSI Ghana", "PSI Ghana, in US$"]].min(axis=1), [-1, .1, .25, np.inf], labels=["stable", "some shift", "big shift"])
drift.to_csv(TABLES / "10_psi.csv")
drift.round(3)

,PSI Kenya test,PSI Ghana,"PSI Ghana, in US$",reading (Ghana)
clue,,,,
loan_type,0.002,17.174,NaN,big shift
duration,0.000,0.642,NaN,big shift
New_versus_Repeat,0.000,0.008,NaN,stable
n_prev_loans,0.000,0.890,NaN,big shift
n_finished,0.001,7.181,NaN,big shift
n_defaults,0.000,0.004,NaN,stable
past_default_rate,0.000,8.887,NaN,big shift
no_known_history,0.000,8.890,NaN,big shift
days_since_last_loan,0.000,1.975,NaN,big shift


In [5]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
bins = np.linspace(0, 12, 49)
for d, lab in [(tr, "Kenya (learned from)"), (gh, "Ghana")]:
    ax[0].hist(np.log(d.Total_Amount), bins=bins, density=True, alpha=.55, label=lab)
    ax[1].hist(np.log(d.Total_Amount / d.usd_rate), bins=np.linspace(-3, 10, 53), density=True, alpha=.55, label=lab)
ax[0].set(title="Loan amount, local currency", xlabel="log(amount): 7 ≈ 1,100, 9 ≈ 8,100", ylabel="share of loans")
ax[1].set(title="Loan amount, US dollars", xlabel="log(US$): 4 ≈ $55, 6 ≈ $400")
ax[0].legend(); plt.tight_layout(); fig.savefig(FIGURES / "10_amounts_kenya_vs_ghana.png", dpi=130)

![](../learnings/figures/10_amounts_kenya_vs_ghana.png)

## 2. Domain classifier: can the clues tell a Ghanaian applicant from a Kenyan one?

A **domain classifier** is a model trained to answer "is this loan from Ghana (1) or Kenya (0)?" from the clues alone. Its ROC-AUC says how different the two groups look:
- **0.5:** it can't tell them apart (the groups look alike);
- **1.0:** it can always tell (the groups don't overlap at all).

We train it on different sets of clues to see **which clues give Ghana away**. Tests keep each customer on one side (5 customer folds).

As a **control**, the same classifier tries to tell Kenya's labelled loans from Kenya's test loans. It should score close to 0.5.

In [6]:
REPAY = ["n_finished", "n_defaults", "past_default_rate", "no_known_history"]   # the outcome clues Ghana can't have
def with_usd(d):
    d = d.copy(); d[AMOUNT_CLUES] = d[AMOUNT_CLUES].div(d.usd_rate, axis=0); return d
SETS = {
    "all 13 clues (money in local currency)": (FEATURES, False),
    "all 13 clues (money in US$)": (FEATURES, True),
    "without loan_type": ([c for c in FEATURES if c != "loan_type"], True),
    "without outcome clues": ([c for c in FEATURES if c not in REPAY], True),
    "without loan_type, outcome clues": ([c for c in FEATURES if c not in REPAY + ["loan_type"]], True),
    "without loan_type, outcome clues, duration": ([c for c in FEATURES if c not in REPAY + ["loan_type", "duration"]], True),
    "loan_type only": (["loan_type"], True),
    "duration only": (["duration"], True),
    "outcome clues only": (REPAY, True),
    "money clues only (local currency)": (AMOUNT_CLUES, False),
    "money clues only (US$)": (AMOUNT_CLUES, True),
    "borrowing counts and speed only": (["n_prev_loans", "days_since_last_loan", "n_loans_last_7d", "n_loans_last_30d"], True),
}
def domain_oof(other, clues, usd, kind="logistic"):
    d = pd.concat([tr.assign(dom=0), other.assign(dom=1)], ignore_index=True)
    d = with_usd(d) if usd else d
    m = make_pipeline(preprocessor(clues), LogisticRegression(max_iter=3000, class_weight="balanced") if kind == "logistic"
                      else LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0, is_unbalance=True))
    cv = list(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(d, d.dom, d.customer_id))
    return d, cross_val_predict(m, d[clues], d.dom, cv=cv, method="predict_proba")[:, 1]
rows, DOM = [], {}
for name, (clues, usd) in SETS.items():
    r = {"clues": name}
    for kind in ["logistic", "LightGBM"]:
        d, p = domain_oof(gh, clues, usd, kind); DOM[name, kind] = (d, p)
        r[f"AUC Ghana vs Kenya ({kind})"] = roc_auc_score(d.dom, p)
    d, p = domain_oof(ke_test, clues, usd, "LightGBM"); r["AUC control: Kenya test vs Kenya (LightGBM)"] = roc_auc_score(d.dom, p)
    rows.append(r)
dom = pd.DataFrame(rows).set_index("clues"); dom.to_csv(TABLES / "10_domain_classifier.csv")
dom.round(3)

,AUC Ghana vs Kenya (logistic),AUC Ghana vs Kenya (LightGBM),AUC control: Kenya test vs Kenya (LightGBM)
clues,,,
all 13 clues (money in local currency),1.000,1.000,0.502
all 13 clues (money in US$),1.000,1.000,0.503
without loan_type,0.992,0.994,0.503
without outcome clues,1.000,1.000,0.499
"without loan_type, outcome clues",0.945,0.973,0.502
"without loan_type, outcome clues, duration",0.939,0.948,0.500
loan_type only,1.000,1.000,0.498
duration only,0.543,0.586,0.500
outcome clues only,0.913,0.913,0.507


## 3. Importance weights: making Kenya look like Ghana

**The idea.** If some Kenyan loans look like Ghanaian ones, we can count those loans more when learning or testing. Then Kenya's data "looks like" Ghana. The weight of a Kenyan loan is

  weight = P(Ghana | clues) ÷ P(Kenya | clues) × (Kenyan loans ÷ Ghanaian loans),

taken from the domain classifier. A Kenyan loan that looks very Ghanaian gets a big weight; a typical Kenyan loan gets almost zero.

**The catch: the effective sample size (ESS).** ESS = (sum of weights)² ÷ (sum of squared weights). It says how many loans the weighted data is "really" worth. If a few loans carry all the weight, ESS collapses, and any weighted score rests on those few.

**Clipping** caps the biggest weights (a percentile cap, or a fixed number; the weights are then re-scaled to average 1, so the largest weight shown can end above the cap). That gives back some ESS, but makes Kenya look less like Ghana. We try several caps.

We use the **logistic** domain classifier on the clues Ghana can actually share with Kenya (no `loan_type`, no outcome clues, money in US$). With all 13 clues the groups barely overlap at all (section 2), so the weights there are meaningless.

In [7]:
def weights(name, kind="logistic", cap=None):
    d, p = DOM[name, kind]
    k = d.dom.to_numpy() == 0
    p = np.clip(p[k], 1e-6, 1 - 1e-6)
    w = p / (1 - p) * (k.sum() / (~k).sum())
    w = w / w.mean()                                   # average weight 1
    if cap is not None:
        w = np.minimum(w, np.quantile(w, cap) if cap < 1 else cap); w = w / w.mean()
    return pd.Series(w, index=tr.index)
def ess(w): return w.sum() ** 2 / (w ** 2).sum()
W_SET = "without loan_type, outcome clues"
CAPS = {"no cap": None, "cap at 99.9th percentile": .999, "cap at 99th percentile": .99, "cap at 50": 50, "cap at 10": 10, "cap at 5": 5}
rows = []
for set_name in [W_SET, "without loan_type, outcome clues, duration", "all 13 clues (money in US$)"]:
    for cap_name, cap in CAPS.items():
        w = weights(set_name, cap=cap)
        rows.append({"clues": set_name, "clipping": cap_name, "ESS (loans)": ess(w), "ESS % of Kenya": ess(w) / len(w),
                     "largest weight": w.max(), "weight held by top 1% of loans": np.sort(w)[::-1][: len(w) // 100].sum() / w.sum()})
wt = pd.DataFrame(rows); wt.to_csv(TABLES / "10_weights.csv", index=False)
d, p = DOM[W_SET, "logistic"]
print(f"Ghanaian loans the classifier is >99% sure are Ghanaian (no Kenyan lookalike): {(p[d.dom == 1] > .99).mean():.1%}")
wt.round(3)

Ghanaian loans the classifier is >99% sure are Ghanaian (no Kenyan lookalike): 11.8%


,clues,clipping,ESS (loans),ESS % of Kenya,largest weight,weight held by top 1% of loans
0,"without loan_type, outcome clues",no cap,18.777,0.000,8810.877,0.989
1,"without loan_type, outcome clues",cap at 99.9th percentile,250.118,0.004,429.479,0.951
2,"without loan_type, outcome clues",cap at 99th percentile,1933.766,0.028,52.758,0.527
3,"without loan_type, outcome clues",cap at 50,343.188,0.005,298.497,0.934
4,"without loan_type, outcome clues",cap at 10,708.321,0.010,136.012,0.849
5,"without loan_type, outcome clues",cap at 5,973.953,0.014,100.213,0.778
6,"without loan_type, outcome clues, duration",no cap,31.178,0.000,6517.508,0.971
7,"without loan_type, outcome clues, duration",cap at 99.9th percentile,314.791,0.005,368.401,0.906
8,"without loan_type, outcome clues, duration",cap at 99th percentile,2428.378,0.035,45.912,0.459
9,"without loan_type, outcome clues, duration",cap at 50,477.546,0.007,227.108,0.869


## 4. Task 1 on real Ghanaian amounts

This is the one place we have real Ghanaian answers. We learn the step 9 amount model on Kenya and predict every Ghanaian loan. We try:
- **currency:** feed cedis in as if they were shillings ("as is"), or convert everything to US dollars;
- **loan type:** keep `loan_type` (Ghana's Type_3 is then simply "unknown" to the model), or drop it;
- **weights:** learn with the importance weights at several caps, or without;
- **naive rules:** "guess Kenya's average", and "the same as your last loan" (for first loans: Ghana's own median, which we're actually not allowed to know, so this rule is a slight cheat).

Scored on the log scale, like step 9. **Bias** is the median of predicted ÷ real: 1.00 is unbiased, 0.5 means it predicts half.

In [8]:
from src.amounts import final_amount_model
def amount_scores(pred_local, d):
    e = np.log(d.Total_Amount.to_numpy()) - np.log(pred_local)
    return {"MAE log": np.abs(e).mean(), "within 2x": (np.abs(e) < np.log(2)).mean(), "bias (predicted / real)": float(np.median(np.exp(-e)))}
def amount_fit(clues, usd, w=None):
    from lightgbm import LGBMRegressor
    X = with_usd(tr) if usd else tr
    y = np.log(tr.Total_Amount / (tr.usd_rate if usd else 1))
    m = make_pipeline(preprocessor(clues), LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=0))
    m.fit(X[clues], y, **({} if w is None else {"lgbmregressor__sample_weight": w.to_numpy()}))
    G = with_usd(gh) if usd else gh
    return np.exp(m.predict(G[clues])) * (gh.usd_rate.to_numpy() if usd else 1)
no_type = [c for c in FEATURES if c != "loan_type"]
rows = [{"option": "guess Kenya's average (local currency)", **amount_scores(np.full(len(gh), np.exp(np.log(tr.Total_Amount).mean())), gh)},
        {"option": "guess Kenya's average (US$)", **amount_scores(np.full(len(gh), np.exp(np.log(tr.Total_Amount / tr.usd_rate).mean())) * gh.usd_rate.to_numpy(), gh)},
        {"option": "same as last loan (first loans: Ghana's median, a slight cheat)", **amount_scores(gh.last_amount.fillna(gh.Total_Amount.median()).to_numpy(), gh)}]
for label, clues, usd in [("model, as is (cedis fed in as shillings)", FEATURES, False), ("model, US$", FEATURES, True),
                          ("model, as is, no loan_type", no_type, False), ("model, US$, no loan_type", no_type, True)]:
    rows.append({"option": label, **amount_scores(amount_fit(clues, usd), gh)})
for cap_name in ["no cap", "cap at 50", "cap at 10"]:
    rows.append({"option": f"model, US$, no loan_type, weighted ({cap_name})", **amount_scores(amount_fit(no_type, True, weights(W_SET, cap=CAPS[cap_name])), gh)})
t1 = pd.DataFrame(rows).set_index("option"); t1.to_csv(TABLES / "10_amount_on_ghana.csv")
t1.round(3)

,MAE log,within 2x,bias (predicted / real)
option,,,
guess Kenya's average (local currency),1.087,0.391,1.760
guess Kenya's average (US$),2.381,0.011,0.127
"same as last loan (first loans: Ghana's median, a slight cheat)",0.522,0.836,0.992
"model, as is (cedis fed in as shillings)",0.453,0.804,1.234
"model, US$",0.509,0.764,0.820
"model, as is, no loan_type",0.507,0.793,0.987
"model, US$, no loan_type",0.626,0.709,0.633
"model, US$, no loan_type, weighted (no cap)",0.736,0.614,0.571
"model, US$, no loan_type, weighted (cap at 50)",0.724,0.618,0.567


**Kenya's own future-loan error, for comparison** (step 9, LightGBM): 0.58 on the log scale, about 1.8×. **By group in Ghana**, with the best option above. A group with a big bias would get limits that are systematically too high or too low.

In [9]:
best_opt = t1.drop(index=[i for i in t1.index if not i.startswith("model")])["MAE log"].idxmin()
print("best model option:", best_opt)
lab = best_opt
usd = "US$" in lab; clues = no_type if "no loan_type" in lab else FEATURES
w = weights(W_SET, cap=CAPS[lab.split("(")[1].rstrip(")")]) if "weighted" in lab else None
pred = amount_fit(clues, usd, w)
gh["_e"] = np.log(gh.Total_Amount) - np.log(pred)
groups = {"loan type": gh.loan_type, "loan length": pd.cut(gh.duration, [0, 7, 30, 90, 1000], labels=["7 days", "8-30", "31-90", "over 90"]),
          "first loan in the data": np.where(gh.n_prev_loans == 0, "first loan", "earlier loans")}
rows = []
for g, s in groups.items():
    for v, e in gh["_e"].groupby(s):
        if len(e) >= 30:
            rows.append({"group": g, "value": v, "loans": len(e), "MAE log": e.abs().mean(), "bias (predicted / real)": float(np.median(np.exp(-e)))})
bygroup = pd.DataFrame(rows); bygroup.to_csv(TABLES / "10_amount_on_ghana_by_group.csv", index=False)
gh.drop(columns="_e", inplace=True)
bygroup.round(3)

best model option: model, as is (cedis fed in as shillings)


,group,value,loans,MAE log,bias (predicted / real)
0,loan type,Type_2,449,1.041,2.819
1,loan type,Type_3,3039,0.330,1.169
2,loan type,Type_8,37,3.471,42.333
3,loan length,7 days,3041,0.331,1.169
4,loan length,over 90,479,1.229,3.125
5,first loan in the data,earlier loans,2863,0.260,1.149
6,first loan in the data,first loan,662,1.290,3.387


## 5. Task 2: how the Kenyan default model treats Ghana

There are no Ghanaian labels, so we can't measure accuracy here. What we *can* see is how the model scores Ghanaian applicants under each option. The **share declined** uses Ghana's cut-off (3.2%) on the calibrated probability (ADR 0007).

The options are the same paths as above, plus what to do with **unseen loan types** (Type_3, Type_8):
- **as is:** the model's one-hot step ignores an unknown type, so these loans get no loan-type push at all;
- **map to Type_1:** treat them as Kenya's most common type (also 7-day, small loans);
- **drop `loan_type`** from the model entirely.

For each option, the **Kenya future-loan lift** shows what it costs where we *can* measure. **Lift** = PR-AUC ÷ default rate, averaged over the 4 future-loan tests. The step 7 model gets about 8.

In [10]:
def calibrated_lr(train, clues, w=None):
    '''Step 7's final recipe (LR + Platt on out-of-fold scores from 5 customer folds), for any clues and optional weights.'''
    X, y = train[clues], train.target.astype(int)
    cv = list(StratifiedGroupKFold(5, shuffle=True, random_state=17).split(X, y, train.customer_id))
    base = lambda: make_pipeline(preprocessor(clues), LogisticRegression(max_iter=2000))
    fp = {} if w is None else {"logisticregression__sample_weight": np.asarray(w)}
    oofp = np.zeros(len(X))
    for a, b in cv:
        oofp[b] = base().fit(X.iloc[a], y.iloc[a], **({} if w is None else {"logisticregression__sample_weight": np.asarray(w)[a]})).predict_proba(X.iloc[b])[:, 1]
    cal = LogisticRegression(max_iter=1000, random_state=17).fit(logit(oofp).reshape(-1, 1), y, **({} if w is None else {"sample_weight": np.asarray(w)}))
    m = base().fit(X, y, **fp)
    return lambda Z: cal.predict_proba(logit(m.predict_proba(Z[clues])[:, 1]).reshape(-1, 1))[:, 1]

# check: with all clues and no weights this is exactly step 7's final model
p_ours, p_step7 = calibrated_lr(tr, FEATURES)(gh), final_model(tr).predict_proba(gh[FEATURES])[:, 1]
assert np.allclose(p_ours, p_step7, atol=1e-6), "should match src/classification_experiments.final_model"

def future_lift(clues, X=None, w=None):
    X = tr if X is None else X; out = []
    for a, b in splits(tr, "time_block"):
        m = make_pipeline(preprocessor(clues), LogisticRegression(max_iter=2000))
        m.fit(X.loc[a, clues], tr.loc[a, "target"], **({} if w is None else {"logisticregression__sample_weight": w[a].to_numpy()}))
        y = tr.loc[b, "target"]; out.append(average_precision_score(y, m.predict_proba(X.loc[b, clues])[:, 1]) / y.mean())
    return np.mean(out)

gh_t1 = gh.assign(loan_type=gh.loan_type.where(gh.loan_type.isin(set(tr.loan_type)), "Type_1"))
first = gh.n_prev_loans.to_numpy() == 0
OPTS = [  # label, clues, money in US$, Ghana rows, weights
    ("as is (step 7 model)", FEATURES, False, gh, None),
    ("money in US$", FEATURES, True, gh, None),
    ("unseen types mapped to Type_1", FEATURES, False, gh_t1, None),
    ("unseen types mapped to Type_1, US$", FEATURES, True, gh_t1, None),
    ("drop loan_type", no_type, False, gh, None),
    ("drop loan_type, US$", no_type, True, gh, None),
    ("drop loan_type, US$, weighted (cap at 10)", no_type, True, gh, weights(W_SET, cap=10)),
    ("drop loan_type, US$, weighted (cap at 50)", no_type, True, gh, weights(W_SET, cap=50)),
]
rows, PD = [], {}
for label, clues, usd, G, w in OPTS:
    T, Gx = (with_usd(tr), with_usd(G)) if usd else (tr, G)
    p = calibrated_lr(T, clues, w)(Gx); PD[label] = p
    dec = p > CUTOFF["Ghana"]
    rows.append({"option": label, "Kenya future-loan lift": future_lift(clues, T, w),
                 "Ghana: average predicted default": p.mean(), "Ghana: share declined": dec.mean(),
                 "declined, first loans": dec[first].mean(), "declined, earlier loans": dec[~first].mean(),
                 **{f"declined, {t}": dec[(G.loan_type if 'mapped' not in label else gh.loan_type).to_numpy() == t].mean() for t in ["Type_3", "Type_2"]}})
t2 = pd.DataFrame(rows).set_index("option")
ke_p = final_model(tr).predict_proba(ke_test[FEATURES])[:, 1]
t2.loc["for comparison: Kenya test applicants, step 7 model, 2.1% cut-off", ["Ghana: average predicted default", "Ghana: share declined"]] = [ke_p.mean(), (ke_p > CUTOFF["Kenya"]).mean()]
t2.to_csv(TABLES / "10_default_model_on_ghana.csv")
t2.round(3)

,Kenya future-loan lift,Ghana: average predicted default,Ghana: share declined,"declined, first loans","declined, earlier loans","declined, Type_3","declined, Type_2"
option,,,,,,,
as is (step 7 model),7.799,0.125,0.474,1.000,0.353,0.390,1.0
money in US$,7.911,0.128,0.467,1.000,0.344,0.382,1.0
unseen types mapped to Type_1,7.799,0.108,0.154,0.601,0.050,0.018,1.0
"unseen types mapped to Type_1, US$",7.911,0.109,0.148,0.601,0.043,0.011,1.0
drop loan_type,6.884,0.087,0.283,1.000,0.117,0.168,1.0
"drop loan_type, US$",6.752,0.085,0.242,1.000,0.067,0.121,1.0
"drop loan_type, US$, weighted (cap at 10)",5.233,0.139,1.000,1.000,1.000,1.000,1.0
"drop loan_type, US$, weighted (cap at 50)",4.657,0.119,0.677,1.000,0.602,0.625,1.0
"for comparison: Kenya test applicants, step 7 model, 2.1% cut-off",NaN,0.018,0.192,NaN,NaN,NaN,NaN


## 6. The best Kenyan stand-ins for Ghana

Ghana's applicants are of two kinds: **first loans** (no earlier loans in the data, 19%) and **earlier loans with no known outcome** (81%). We find Kenyan loans of the same two kinds and score the step 7 model on them, on future loans. These are the closest measurable guesses of how well the model ranks Ghanaian risk, *if Ghanaians behaved like those Kenyans*.

The interval is a 95% customer bootstrap (1,000 re-draws).

In [11]:
o = []
for a, b in splits(tr, "time_block"):
    m = make_pipeline(preprocessor(FEATURES), LogisticRegression(max_iter=2000)).fit(tr.loc[a, FEATURES], tr.loc[a, "target"])
    o.append(pd.DataFrame({"fold": len(o), "p": m.predict_proba(tr.loc[b, FEATURES])[:, 1]}, index=b))
o = pd.concat(o).join(tr[["target", "customer_id", "n_prev_loans", "no_known_history"]])
def lift(g): return average_precision_score(g.target, g.p) / g.target.mean()
def boot_lift(g, n=1000, seed=0):
    rng, cs = np.random.default_rng(seed), g.customer_id.unique(); by = {c: i for c, i in g.groupby("customer_id").indices.items()}
    vals = []
    for _ in range(n):
        idx = np.concatenate([by[c] for c in rng.choice(cs, len(cs))]); s = g.iloc[idx]
        if s.target.sum() > 0: vals.append(lift(s))
    return np.percentile(vals, [2.5, 97.5])
groups = {"all future loans": o, "first loans (like Ghana's 19%)": o[o.n_prev_loans == 0],
          "earlier loans, no known outcome (like Ghana's 81%)": o[(o.n_prev_loans > 0) & (o.no_known_history == 1)],
          "earlier loans, known outcomes (no Ghanaian has these)": o[o.no_known_history == 0]}
rows = []
for name, g in groups.items():
    lo, hi = boot_lift(g)
    rows.append({"Kenyan loans": name, "loans": len(g), "defaults": int(g.target.sum()), "default rate": g.target.mean(),
                 "lift (pooled)": lift(g), "95% low": lo, "95% high": hi})
stand = pd.DataFrame(rows).set_index("Kenyan loans"); stand.to_csv(TABLES / "10_kenyan_standins.csv")
stand.round(3)

,loans,defaults,default rate,lift (pooled),95% low,95% high
Kenyan loans,,,,,,
all future loans,54463,988,0.018,7.072,6.113,8.429
first loans (like Ghana's 19%),2514,275,0.109,2.497,2.061,3.039
"earlier loans, no known outcome (like Ghana's 81%)",1436,69,0.048,6.464,4.509,10.403
"earlier loans, known outcomes (no Ghanaian has these)",50513,644,0.013,5.894,4.815,7.836


## 7. How many labelled Ghanaian loans would NkwaPay need?

Once Ghanaian loans start falling due, NkwaPay can measure the model there. But with only a few defaults that measure is very noisy. We imitate this with Kenyan future loans: draw a random sample of *n* loans many times, compute the lift each time, and see how wide the spread is.

In [12]:
rng, rows = np.random.default_rng(0), []
for n in [500, 1000, 2000, 5000, 10000, 20000]:
    vals, dfl = [], []
    for _ in range(300):
        s = o.iloc[rng.choice(len(o), n, replace=False)]
        if s.target.sum() >= 2: vals.append(lift(s)); dfl.append(s.target.sum())
    lo, hi = np.percentile(vals, [2.5, 97.5])
    rows.append({"labelled loans": n, "defaults (typical)": int(np.median(dfl)), "lift 95% low": lo, "lift 95% high": hi, "width": hi - lo})
need = pd.DataFrame(rows).set_index("labelled loans"); need.to_csv(TABLES / "10_sample_size.csv")
need.round(2)

,defaults (typical),lift 95% low,lift 95% high,width
labelled loans,,,,
500,9,3.02,32.81,29.79
1000,18,3.22,21.52,18.31
2000,36,4.47,15.17,10.70
5000,90,4.95,10.94,5.99
10000,182,5.73,9.14,3.41
20000,364,6.05,8.52,2.47


## 8. What this means

1. **Ghana is not "Kenya in another currency".** The domain classifier tells them apart almost perfectly (ROC-AUC 1.00 with all clues), while the Kenyan control scores 0.50. Most of the gap comes from the clues Ghana *can't* share:
   - `loan_type`: 87% of Ghanaian loans use a type never seen in Kenya;
   - the outcome clues: no Ghanaian has a known repayment.

   Even without those clues, the AUC is still 0.95–0.97.
2. **Importance weighting can't fix this.** The weighted Kenyan data is worth about 19 loans (ESS) without a cap, and at best about 2,800 loans (4%) with the strongest cap. Every weighted model was worse:
   - the Kenyan lift falls from 6.9 to 4.7–5.2;
   - the decline rate in Ghana jumps to 68–100%.

   **We don't use weights.**
3. **The default model, as it stands, would decline 47% of Ghanaian applicants**, against 19% in Kenya. That includes every first-time borrower and every Type_2 loan. The type effect is learned from 74 Kenyan Type_2 loans, 61% of which defaulted. Mapping unseen types to Type_1 brings the decline rate to 15%. Dropping `loan_type` gives 24–28%, and costs the Kenyan lift 7.8 → 6.9. None of these can be checked without Ghanaian outcomes.
4. **The amount model transfers surprisingly well** on real Ghanaian amounts, if cedis are fed in as they are: the log error is 0.45, better than Kenya's own 0.58. It mostly learns "about the size of your recent loans". Converting to dollars is *worse* (0.51), and weights are much worse (0.72–0.75). But it **over-predicts** for:
   - first loans: 3.4×;
   - loans over 90 days: 3.1×;
   - Type_2: 2.8×.
5. **The closest Kenyan stand-ins** suggest the model ranks Ghanaian-like borrowers less well:
   - first loans: lift 2.5 (95%: 2.1–3.0);
   - earlier loans with no known outcome: lift 6.5 (4.5–10.4, only 69 defaults);
   - all future loans, for comparison: 7.1.
6. **Evidence needed before trusting it in Ghana:** about **5,000 labelled Ghanaian loans** (~90 defaults) to know the lift within about ±3, and 10,000 (~180 defaults) for about ±1.7.

The full write-up is in [learnings/10](../learnings/10_transfer.md), and the decision in [ADR 0019](../docs/adr/0019-kenya-to-ghana-transfer.md).

---

# Part 2 · Fairness (step 11, Liza)

*To be added by Liza. Useful inputs from Part 1: `PD` (calibrated default probabilities for Ghana under each option), `gh`, `tr`, and the tables in `learnings/tables/10_*.csv`.*